https://machinelearningmastery.com/how-to-grid-search-hyperparameters-for-pytorch-models/

In [1]:
import numpy as np  # Loading data
import torch
import torch.nn.functional as F
import torchmetrics
from torch.utils.data import DataLoader, TensorDataset  # Tensor dataset and construct dataloader
import torch.optim as optim  # Optimization function
import torch.nn as nn  # Call Layers and construct model
from torch.optim.lr_scheduler import ReduceLROnPlateau  # Adjust Learning rate during training
from torchmetrics.functional import precision, recall  # PyTorch performance metrics
from torchmetrics.classification import MulticlassPrecision, MulticlassRecall, MulticlassF1Score  # Multi-class metrics
from sklearn import metrics  # Model performance metrics
import pickle
import os
import cv2  # Read image; Resize image function
from tqdm import tqdm  # Instantly make loops show progress
import matplotlib.image  # Save array as image
import shutil
from huggingface_hub import hf_hub_download  # For loading pre-trained models
from transformers import ViTImageProcessor, ViTForImageClassification  # ViTFeatureExtractor
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler  # Balancing multi-class dataset
from imblearn.under_sampling import RandomUnderSampler
import pandas as pd
from skorch import NeuralNetClassifier # Use PyTorch Models in scikit-learn
from sklearn.model_selection import GridSearchCV # Grid Search

In [2]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print(device)

cuda


# Data

In [3]:
# Load dataset
with open('C:/Users/101194208/Desktop/Die-Cast Ensemble/Data/data_oversample_usingViT.pkl', 'rb') as file:
    loaded_data = pickle.load(file)

# Access the Loaded variables
X_train, X_val, X_test = loaded_data['var1'], loaded_data['var2'], loaded_data['var3']
y_train, y_val, y_test = loaded_data['var4'], loaded_data['var5'], loaded_data['var6']

# X_train = X_train.astype('float32');print(type(X_train))
X_train = torch.Tensor(X_train)
y_train = torch.LongTensor(y_train)

# import cv2
# import numpy as np
# new_height = 224
# new_width = 224
# data = X_train
# resized_data = np.empty((data.shape[0], data.shape[1], new_height, new_width), dtype=data.dtype)
# for i in range(data.shape[0]):
#     # Extract a single 3D image (channels, height, width)
#     img_3d = data[i]
#     # Transpose to (height, width, channels) for OpenCV
#     img_cv2 = np.transpose(img_3d, (1, 2, 0))
#     # Resize the image
#     resized_img_cv2 = cv2.resize(img_cv2, (new_width, new_height), interpolation=cv2.INTER_LINEAR)
#     # Transpose back to (channels, height, width)
#     resized_3d = np.transpose(resized_img_cv2, (2, 0, 1))
#     # Store in the new array
#     resized_data[i] = resized_3d
# X_train = torch.Tensor(resized_data)
# y_train = torch.LongTensor(y_train)

# PyTorch Classifier

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = self._conv_layer_set(3, 16)
        self.conv2 = self._conv_layer_set(16, 32)
        self.conv3 = self._conv_layer_set(32, 64)
        
        # Adjust the first fully connected layer size based on input dimension
        self.fc1 = nn.Linear(26 * 26 * 64, 128)  # Ensure correct input size
        self.fc2 = nn.Linear(128, 3)  # Change output neurons to 3 for multi-class

        self.relu = nn.LeakyReLU()
        self.conv1_bn = nn.BatchNorm2d(16)
        self.conv2_bn = nn.BatchNorm2d(32)
        self.conv3_bn = nn.BatchNorm2d(64)
        self.fc1_bn = nn.BatchNorm1d(128)
        self.drop = nn.Dropout(p=0.3)

    def _conv_layer_set(self, in_channels, out_channels):
        conv_layer = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(3, 3), stride=1, padding=0),
            nn.LeakyReLU(),
            nn.MaxPool2d(kernel_size=(2, 2), stride=2)
        )
        return conv_layer

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv1_bn(x)
        x = self.conv2(x)
        x = self.conv2_bn(x)
        x = self.conv3(x)
        x = self.conv3_bn(x)

        x = x.view(x.size(0), -1)  # Flatten before FC layer
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc1_bn(x)
        x = self.drop(x)
        x = self.fc2(x)

        return x  # Return logits (no softmax, use CrossEntropyLoss)

# Create Model with skorch

In [5]:
model = NeuralNetClassifier(
    CNN,
    criterion = nn.CrossEntropyLoss(),
    optimizer = optim.Adam,
    verbose = False,
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

# Grid Search

In [6]:
param_grid = {
    'batch_size': [32, 64],
    'max_epochs': [20, 30, 40],
    'optimizer__lr': [0.0001, 0.001, 0.01, 0.1]}

In [7]:
%%time

grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=1, cv=3)
grid_result = grid.fit(X_train, y_train)

CPU times: total: 3h 3min 58s
Wall time: 23min 17s


In [8]:
# summarize results
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

Best: 0.859497 using {'batch_size': 64, 'max_epochs': 30, 'optimizer__lr': 0.0001}
0.613515 (0.020870) with: {'batch_size': 32, 'max_epochs': 20, 'optimizer__lr': 0.0001}
0.429337 (0.041430) with: {'batch_size': 32, 'max_epochs': 20, 'optimizer__lr': 0.001}
0.333951 (0.001165) with: {'batch_size': 32, 'max_epochs': 20, 'optimizer__lr': 0.01}
0.334569 (0.002039) with: {'batch_size': 32, 'max_epochs': 20, 'optimizer__lr': 0.1}
0.648125 (0.188774) with: {'batch_size': 32, 'max_epochs': 30, 'optimizer__lr': 0.0001}
0.449114 (0.114853) with: {'batch_size': 32, 'max_epochs': 30, 'optimizer__lr': 0.001}
0.333333 (0.000291) with: {'batch_size': 32, 'max_epochs': 30, 'optimizer__lr': 0.01}
0.336012 (0.004110) with: {'batch_size': 32, 'max_epochs': 30, 'optimizer__lr': 0.1}
0.673671 (0.047757) with: {'batch_size': 32, 'max_epochs': 40, 'optimizer__lr': 0.0001}
0.475484 (0.081654) with: {'batch_size': 32, 'max_epochs': 40, 'optimizer__lr': 0.001}
0.332921 (0.000771) with: {'batch_size': 32, 'max_